In [ ]:
import os
import json
import ollama
from dotenv import load_dotenv
from IPython.display import Markdown, display
from scraper import fetch_website_links, fetch_website_contents

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OLLAMA_API_KEY')

if api_key:
    print("API key looks good so far")
else:
    print("looks good so far!")
    
MODEL = 'llama3.2'

API key looks good so far


In [3]:
links = fetch_website_links("https://fordive.id/")
links

['#content',
 'https://fordive.id/',
 'https://fordive.id/shop/',
 'https://fordive.id/mission/',
 'https://fordive.id/about-us/',
 'https://fordive.id/distributor/',
 'https://fordivescentfinder.id/quiz',
 'https://fordive.id/home-2/mission/',
 'https://fordive.id/shop/revolt/',
 'https://fordive.id/shop/garden-breeze/',
 'https://fordive.id/shop/shelby/',
 'https://fordive.id/shop/feeling-good/',
 'https://fordive.id/shop/',
 'https://fordive.id',
 'https://fordive.id/aboutus/',
 'https://fordive.id/fordive/mission/',
 'https://fordive.id/fordive/shop/',
 'https://shopee.co.id/fordive',
 'https://www.tokopedia.com/fordiveperfume',
 'https://www.lazada.co.id/shop/fordive',
 'https://www.tiktok.com/@fordive.id',
 'https://fordive.id/distributor/',
 'https://www.instagram.com/fordive.id/',
 'https://www.tiktok.com/@fordive.id',
 'https://x.com/Fordiveperfume',
 'https://www.facebook.com/Fordive.idn/']

In [4]:
link_system_prompts = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://fordive.id/about-us/"},
        {"type": "careers page", "url": "https://dribbble.com/tags/careers-page"}
    ]
}   
"""

In [5]:
def get_links_user_prompts(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
get_links_user_prompts("https://fordive.id/")

'\nHere is the list of links on the website https://fordive.id/ -\nPlease decide which of these are relevant web links for a brochure about the company, \nrespond with the full https URL in JSON format.\nDo not include Terms of Service, Privacy, email links.\n\nLinks (some might be relative links):\n\n#content\nhttps://fordive.id/\nhttps://fordive.id/shop/\nhttps://fordive.id/mission/\nhttps://fordive.id/about-us/\nhttps://fordive.id/distributor/\nhttps://fordivescentfinder.id/quiz\nhttps://fordive.id/home-2/mission/\nhttps://fordive.id/shop/revolt/\nhttps://fordive.id/shop/garden-breeze/\nhttps://fordive.id/shop/shelby/\nhttps://fordive.id/shop/feeling-good/\nhttps://fordive.id/shop/\nhttps://fordive.id\nhttps://fordive.id/aboutus/\nhttps://fordive.id/fordive/mission/\nhttps://fordive.id/fordive/shop/\nhttps://shopee.co.id/fordive\nhttps://www.tokopedia.com/fordiveperfume\nhttps://www.lazada.co.id/shop/fordive\nhttps://www.tiktok.com/@fordive.id\nhttps://fordive.id/distributor/\nhttps

In [14]:
def select_relevant_links(url):
    response = ollama.chat(
        model=MODEL,
        messages=[
            {'role':'system','content':link_system_prompts},
            {'role':'user','content':get_links_user_prompts(url)}
        ],
        format='json'
    )
    results = response['message']['content']
    links = json.loads(results)
    return links

In [15]:
select_relevant_links("https://fordive.id/")

{'links': [{'type': 'about page', 'url': 'https://fordive.id/about-us/'},
  {'type': 'company page', 'url': 'https://fordive.id'},
  {'type': 'shop page', 'url': 'https://fordive.id/shop/'},
  {'type': 'distributor page', 'url': 'https://fordive.id/distributor/'}]}

In [16]:
def fetch_page_and_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n ## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [17]:
fetch_page_and_relevant_links("https://fordive.id/")

'## Landing Page:\n\nFordive\n\nSkip to content\nShop\nMission\nAbout Us\nDistributor\nFind Your Scent\nX\nHome\nLove More\nLive More\nSince the first day we believe that Fordive is not Just a Product, we are a brand with vision.\n“Always have a positive impact on society”\n. So we decided our mission to deliver the importance of self love and also share it with others.\nLearn More\nInfo Update\nNew Launch\nGarden Breeze\nFresh, Green, Floral\nLearn More\nBest Seller\nShelby\nCool, Fruity, Aromatic\nLearn More\nBest Seller\nFeeling Good\nSweet, Fruity, Floral\nLearn More\nAbout Our Product\nWhy Fordive?\nWe strongly believe that\n“The greatest love doesn’t result from what we get, but from what we give”\n.\u2028Therefore, It’s our main mission to put our Love and Passion in our products.\n01\nLong-Lasting Premium Fragrance\nWe strongly believe that\xa0“The greatest love doesn’t result from what we get, but from what we give”.\u2028Therefore, It’s our main mission to put our Love and Pa

In [18]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [25]:
def get_brochure_user_prompt(company_name,url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [31]:
get_brochure_user_prompt("Fordive","https://fordive.id/")

"\nYou are looking at a company called: Fordive\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nFordive\n\nSkip to content\nShop\nMission\nAbout Us\nDistributor\nFind Your Scent\nX\nHome\nLove More\nLive More\nSince the first day we believe that Fordive is not Just a Product, we are a brand with vision.\n“Always have a positive impact on society”\n. So we decided our mission to deliver the importance of self love and also share it with others.\nLearn More\nInfo Update\nNew Launch\nGarden Breeze\nFresh, Green, Floral\nLearn More\nBest Seller\nShelby\nCool, Fruity, Aromatic\nLearn More\nBest Seller\nFeeling Good\nSweet, Fruity, Floral\nLearn More\nAbout Our Product\nWhy Fordive?\nWe strongly believe that\n“The greatest love doesn’t result from what we get, but from what we give”\n.\u2028Therefore, It’s our main mission to put our Love and Passion i

In [ ]:
def create_brochure_company(company_name, url):
    stream = ollama.chat(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': brochure_system_prompt},
            {'role': 'user', 'content': get_brochure_user_prompt(company_name, url)}
        ],
        stream=True 
    )
    
    response = "" 
    display_handle = display(Markdown(""), display_id=True)
    
    for chunk in stream:
        content = chunk['message']['content'] or ''
        response += content
        
        display_handle.update(Markdown(response))

In [43]:
create_brochure_company("Fordive","https://fordive.id/")

# Welcome to Fordive: Where Love Meets Luxury

At Fordive, we believe that the greatest love doesn't come from what we get, but from what we give. Our mission is to deliver high-quality perfumes that not only tantalize your senses but also spread positivity and self-love.

## Our Story

Our journey began in May 2020 with a simple idea: to create an affordable perfume brand that delivers international standards. We vowed to make it happen, driven by our passion for providing exceptional products and services. From the start, we've been committed to making a positive impact on society, promising to spare 10% of our sales to support the "Disabled Children Education Foundation."

## Our Values

*   **Love at First Sight**: We believe that every human is unique, and your voice matters. Your ideas help us create better products!
*   **Natural and Ethical Fragrance**: We ensure only the best and safest ingredients are used in our perfumes.
*   **Premium Quality**: We're committed to delivering long-lasting, high-quality fragrances that exceed your expectations.

## Our Products

    *   Men Series: Indulge in the epitome of olfactory luxury with our men's collection, featuring fresh, green, and citrus scents like Revolt.
    *   Woman Series: Treat yourself to elegance with our woman's collection, showcasing sweet, fruity, and floral aromas like Shelby.
    *   Unisex: Explore our unisex range, perfect for those who prefer a subtle yet sophisticated scent.

## Join the Fordive Family

At Fordive, we're not just a perfume brand – we're a community of individuals united by our passion for self-love and luxury. Whether you're a customer, investor, or future team member, join us on this journey and let's create something amazing together!

### Let's Connect:

*   Instagram: [link]
*   TikTok: [link]
*   Facebook: [link]

    Join the conversation, share your thoughts, and let's spread love and positivity!